# Legal Tech API (Kaggle Notebook Version)
This notebook launches a FastAPI server for legal Q&A and summarization using Gemma and Legal LED models, and exposes it via ngrok for external access.

In [ ]:
# Install dependencies
!pip install fastapi uvicorn transformers torch accelerate pydantic[dotenv] python-multipart pyngrok --quiet

In [ ]:
# Download models from Hugging Face
from transformers import AutoModelForCausalLM, AutoTokenizer, LEDForConditionalGeneration
import os

# Gemma
gemma_dir = 'google_gemma-3-1b-it-qat-q4_0-unquantized'
if not os.path.exists(gemma_dir):
    tokenizer = AutoTokenizer.from_pretrained('google/gemma-3-1b-it-qat-q4_0-unquantized')
    model = AutoModelForCausalLM.from_pretrained('google/gemma-3-1b-it-qat-q4_0-unquantized')
    tokenizer.save_pretrained(gemma_dir)
    model.save_pretrained(gemma_dir)

# Legal LED
led_dir = 'legal_led_finetuned'
if not os.path.exists(led_dir):
    tokenizer = AutoTokenizer.from_pretrained('Dharil/legal-led-base-16384-finetuned-legal-tech-IN')
    model = LEDForConditionalGeneration.from_pretrained('Dharil/legal-led-base-16384-finetuned-legal-tech-IN')
    tokenizer.save_pretrained(led_dir)
    model.save_pretrained(led_dir)

In [ ]:
# Write FastAPI app code
with open('app.py', 'w') as f:
    f.write('''
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from fastapi.middleware.cors import CORSMiddleware
from transformers import AutoModelForCausalLM, AutoTokenizer, LEDForConditionalGeneration
import torch
import os
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("legal_tech_api")

# Gemma Model
class GemmaModel:
    def __init__(self, model_dir):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.tokenizer = AutoTokenizer.from_pretrained(model_dir)
        self.model = AutoModelForCausalLM.from_pretrained(model_dir).to(self.device)
        self.current_judgment = None
    def set_judgment(self, judgment_text):
        self.current_judgment = judgment_text
    def preprocess(self, question):
        if not self.current_judgment:
            raise ValueError("No judgment text has been set.")
        return f"Judgment: {self.current_judgment}\nQuestion: {question}"
    def infer(self, input_text):
        inputs = self.tokenizer(input_text, return_tensors="pt").to(self.model.device)
        with torch.no_grad():
            outputs = self.model.generate(**inputs, max_new_tokens=256)
        return self.tokenizer.decode(outputs[0], skip_special_tokens=True)
    def postprocess(self, raw_output):
        answer = raw_output.split("Sources:")[0].strip()
        sources = raw_output.split("Sources:")[1].strip().split(',') if "Sources:" in raw_output else []
        return {"answer": answer, "sources": sources}

# Legal LED Model
class LegalLEDModel:
    def __init__(self, model_dir):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.tokenizer = AutoTokenizer.from_pretrained(model_dir)
        self.model = LEDForConditionalGeneration.from_pretrained(model_dir).to(self.device)
    def summarize(self, text):
        inputs = self.tokenizer(text, return_tensors="pt", truncation=True, max_length=4096).to(self.model.device)
        with torch.no_grad():
            outputs = self.model.generate(**inputs, max_new_tokens=256)
        return self.tokenizer.decode(outputs[0], skip_special_tokens=True)

app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_credentials=True, allow_methods=["*"], allow_headers=["*"])

gemma = GemmaModel('google_gemma-3-1b-it-qat-q4_0-unquantized')
legal_led = LegalLEDModel('legal_led_finetuned')

class SetJudgmentRequest(BaseModel):
    judgment_text: str
class QuestionRequest(BaseModel):
    question: str
class ChatResponse(BaseModel):
    answer: str
    sources: list[str]
class SummaryRequest(BaseModel):
    judgment_text: str
class SummaryResponse(BaseModel):
    answer: str

@app.get("/")
def root():
    return {"answer": "Welcome to the Legal Tech Chat API!"}

@app.post("/set-judgment")
def set_judgment(request: SetJudgmentRequest):
    gemma.set_judgment(request.judgment_text)
    return {"answer": "Judgment text has been set successfully"}

@app.post("/chat", response_model=ChatResponse)
def chat(request: QuestionRequest):
    input_text = gemma.preprocess(request.question)
    raw_output = gemma.infer(input_text)
    result = gemma.postprocess(raw_output)
    return ChatResponse(**result)

@app.post("/summary", response_model=SummaryResponse)
def summarize(request: SummaryRequest):
    answer = legal_led.summarize(request.judgment_text)
    return SummaryResponse(answer=answer)
''')

In [ ]:
# Start FastAPI server in background
import uvicorn
from threading import Thread
def run():
    uvicorn.run('app:app', host='0.0.0.0', port=7860)
Thread(target=run, daemon=True).start()

In [ ]:
# Setup ngrok tunnel
from pyngrok import ngrok
public_url = ngrok.connect(7860, "http").public_url
print(f'✅ FastAPI server is live at: {public_url}')

## Example: Test the API endpoints
You can use the following code to test the endpoints from within the notebook.

In [ ]:
import requests
# Set the base URL to the ngrok public URL
BASE_URL = public_url

# 1. Set judgment text
resp = requests.post(f'{BASE_URL}/set-judgment', json={"judgment_text": "The court ruled in favor of the plaintiff, awarding $50,000 in damages for breach of contract."})
print('Set judgment:', resp.json())

# 2. Ask a question
resp = requests.post(f'{BASE_URL}/chat', json={"question": "What was the basis for the damages award?"})
print('Chat:', resp.json())

# 3. Summarize
resp = requests.post(f'{BASE_URL}/summary', json={"judgment_text": "The court ruled in favor of the plaintiff, awarding $50,000 in damages for breach of contract under Section 12.3 of the Commercial Code. The defendant failed to deliver goods as specified in the agreement dated 2023-01-15."})
print('Summary:', resp.json())